# Answer Sheet Evaluation

This notebook evaluates digitized student examination papers against the answer key.

## Dataset layout (read-only)

- `A Dataset of Digitized Student Examination Papers,/A Dataset of Digitized Student Examination Papers,/` contains the raw dataset.
- `Student_Pdf/`: raw student answer-sheet PDFs.
- `Corrected_Pdf/`: corrected/graded student PDFs.
- `Question.txt`: examination questions.
- `answerkey.txt`: answer key.
- `Student_MCQ.csv`: student MCQ responses.
- `Teacher_manual_marks_Anonymized.csv`: teacher marks.
- `README.md`: dataset documentation.

> All exploration below is **read-only**. No original dataset files are modified, renamed, moved, or deleted.

## 1. Setup and Imports

Import all required packages. Standard-library modules (`json`, `pathlib`, `re`) are always available; third-party packages are wrapped in `try/except` so the notebook can report which ones are missing.

In [1]:
import json
import re
from pathlib import Path

package_status = {}

# Third-party packages
try:
    import cv2
    package_status["opencv-python (cv2)"] = True
except Exception as e:
    package_status["opencv-python (cv2)"] = False

try:
    import fitz  # PyMuPDF
    package_status["PyMuPDF (fitz)"] = True
except Exception as e:
    package_status["PyMuPDF (fitz)"] = False

try:
    from PIL import Image
    package_status["Pillow (PIL)"] = True
except Exception as e:
    package_status["Pillow (PIL)"] = False

try:
    import pandas as pd
    package_status["pandas"] = True
except Exception as e:
    package_status["pandas"] = False

try:
    import numpy as np
    package_status["numpy"] = True
except Exception as e:
    package_status["numpy"] = False

try:
    import matplotlib.pyplot as plt
    package_status["matplotlib"] = True
except Exception as e:
    package_status["matplotlib"] = False

try:
    from dotenv import load_dotenv
    package_status["python-dotenv (dotenv)"] = True
except Exception as e:
    package_status["python-dotenv (dotenv)"] = False

print("Imports attempted. See the next cell for a status summary.")

c:\Users\Shikhar\OneDrive\Desktop\assessment\venv\Lib\site-packages\fitz\__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


Imports attempted. See the next cell for a status summary.


## 2. Dependency Availability and Installation Command

Run the cell below to print which packages are available and which are missing. If any are missing, copy the generated `pip install` command to install them.

> Note: `json`, `pathlib`, and `re` are part of the Python standard library and do not need installation.

In [2]:
import pandas as pd

# Build status table
status_rows = [
    {"package": pkg, "available": "Yes" if ok else "No"}
    for pkg, ok in package_status.items()
]
status_df = pd.DataFrame(status_rows)
print(status_df.to_string(index=False))

missing = [pkg for pkg, ok in package_status.items() if not ok]

if missing:
    print("\nMissing packages:", missing)
    install_cmd = "pip install " + " ".join([
        "opencv-python",
        "pymupdf",
        "Pillow",
        "pandas",
        "numpy",
        "matplotlib",
        "python-dotenv",
    ])
    print("\nInstallation command:")
    print(install_cmd)
else:
    print("\nAll third-party packages are available.")

               package available
   opencv-python (cv2)       Yes
        PyMuPDF (fitz)       Yes
          Pillow (PIL)       Yes
                pandas       Yes
                 numpy       Yes
            matplotlib       Yes
python-dotenv (dotenv)       Yes

All third-party packages are available.


## 3. Automatic Dataset Directory Detection

The cell below automatically locates the dataset directory (tolerant of the duplicated folder name) and inventories every file inside it. If dependencies are missing, this cell still runs using only standard-library modules.

In [3]:
from pathlib import Path

WORKSPACE_ROOT = Path(".")
DATASET_NAME = "A Dataset of Digitized Student Examination Papers,"

def find_dataset_root(root: Path):
    """Locate the inner dataset folder even when the name is duplicated."""
    candidates = []
    for dirpath, dirnames, _ in root.walk():
        dirpath = Path(dirpath)
        for d in dirnames:
            if d == DATASET_NAME:
                candidates.append(dirpath / d)
    if not candidates:
        return None
    return sorted(candidates, key=lambda p: len(p.parts), reverse=True)[0]

dataset_root = find_dataset_root(WORKSPACE_ROOT)

if dataset_root is None:
    print("Dataset directory not found automatically.")
    print("Expected name:", DATASET_NAME)
else:
    files_only = [f for f in sorted(dataset_root.rglob("*")) if f.is_file()]
    student_pdf_dir = dataset_root / "Student_Pdf"
    corrected_pdf_dir = dataset_root / "Corrected_Pdf"

    print(f"Detected dataset root : {dataset_root}")
    print(f"Total files           : {len(files_only)}")
    print(f"Student_Pdf count     : {len(list(student_pdf_dir.glob('*.pdf'))) if student_pdf_dir.exists() else 0}")
    print(f"Corrected_Pdf count   : {len(list(corrected_pdf_dir.glob('*.pdf'))) if corrected_pdf_dir.exists() else 0}")


Detected dataset root : A Dataset of Digitized Student Examination Papers,\A Dataset of Digitized Student Examination Papers,
Total files           : 106
Student_Pdf count     : 50
Corrected_Pdf count   : 50


## 4. Dataset Exploration

Read the key dataset files and display statistics. Everything here is read-only.

- `README.md` — dataset documentation
- `answerkey.txt` — ground-truth answer key
- `Question.txt` — examination questions
- `Teacher_manual_marks_Anonymized.csv` — teacher-assigned itemized marks
- `Student_Pdf/` and `Corrected_Pdf/` — raw and corrected student PDFs

In [4]:
import re
import pandas as pd

if 'dataset_root' in globals() and dataset_root is not None:
    ak_path = dataset_root / "answerkey.txt"
    if ak_path.exists():
        lines = [l.split(",") for l in ak_path.read_text(encoding="utf-8").strip().splitlines()[1:] if l.strip()]
        mcq_count = sum(1 for r in lines if len(r) >= 2 and r[1].strip() == "MCQ")
        short_count = sum(1 for r in lines if len(r) >= 2 and r[1].strip() == "Short_Answer")
        print(f"Answer key : {len(lines)} questions ({mcq_count} MCQ, {short_count} Short_Answer)")

    q_path = dataset_root / "Question.txt"
    if q_path.exists():
        q_nums = re.findall(r'^(\d+)\.\s', q_path.read_text(encoding="utf-8"), re.MULTILINE)
        print(f"Questions  : {len(q_nums)} (Q{q_nums[0]}..Q{q_nums[-1]})")

    marks_path = dataset_root / "Teacher_manual_marks_Anonymized.csv"
    if marks_path.exists():
        marks_df = pd.read_csv(marks_path)
        print(f"Teacher CSV: {marks_df.shape[0]} students x {marks_df.shape[1]} columns")
        print(marks_df[["Student_ID", "MCQ mark", "2 marks", "total"]].head(5).to_string(index=False))
else:
    print("Dataset root not detected; run Section 3 first.")


Answer key : 35 questions (20 MCQ, 15 Short_Answer)
Questions  : 35 (Q1..Q35)
Teacher CSV: 50 students x 40 columns
Student_ID  MCQ mark  2 marks  total
 Student_1        12     21.0   33.0
 Student_2        10     13.0   23.0
 Student_3        11     13.0   24.0
 Student_4        15     21.0   36.0
 Student_5        11     15.0   26.0


## 5. Dataset File Summary Table

A compact table of detected files, grouped by type and folder.

In [5]:
import pandas as pd

if 'dataset_root' in globals() and dataset_root is not None:
    rows = []
    for f in sorted(x for x in dataset_root.rglob("*") if x.is_file()):
        rel = f.relative_to(dataset_root)
        folder = rel.parts[0] if len(rel.parts) > 1 else "root"
        rows.append({"category": folder, "extension": f.suffix.lower(), "size_bytes": f.stat().st_size})
    df = pd.DataFrame(rows)
    grouped = df.groupby(["category", "extension"], as_index=False).agg(
        files=("size_bytes", "count"),
        total_mb=("size_bytes", lambda s: round(s.sum() / (1024 * 1024), 2))
    )
    print(grouped.to_string(index=False))
else:
    print("Dataset root not detected; run Section 3 first.")


     category extension  files  total_mb
Corrected_Pdf      .pdf     50     86.46
  Student_Pdf      .pdf     50     83.47
         root      .csv      2      0.01
         root       .md      1      0.01
         root      .txt      3      0.01


## 6. PDF Ingestion and Image Processing

Convert student PDF pages (configured via `MAX_STUDENTS = 5`) into images using **PyMuPDF** and preprocess with **OpenCV** (grayscale, bounded skew correction, denoising, CLAHE).
Processed pages are saved to `outputs/processed_pages/` with per-student checkpoints so ingestion resumes cleanly if interrupted without holding raw images in RAM.

> Read-only: no original dataset files are modified.

In [6]:
import json, re, cv2, numpy as np
from pathlib import Path

try:
    import pymupdf as fitz
except ImportError:
    import fitz

def _student_sort_key(path):
    m = re.search(r"(\d+)", path.stem)
    return (0, int(m.group(1))) if m else (1, path.stem)

def detect_skew(gray, max_deg=3.0):
    """Detect skew angle capped at +-max_deg to avoid false rotations on handwriting."""
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    edges = cv2.Canny(blur, 50, 150, apertureSize=3)
    lines = cv2.HoughLinesP(edges, 1, np.pi / 180, 80, minLineLength=100, maxLineGap=15)
    if lines is None:
        return 0.0
    angles = []
    for l in lines:
        x1, y1, x2, y2 = l.reshape(4)
        dx, dy = x2 - x1, y2 - y1
        if dx == 0:
            continue
        deg = float(np.degrees(np.arctan2(dy, dx)))
        if abs(deg) < 45:
            angles.append(deg)
    if len(angles) < 5:
        return 0.0
    angle = round(float(np.median(angles)), 2)
    return max(-max_deg, min(max_deg, angle))

pdf_ingestion_results = []
PROC_DIR = Path("outputs/processed_pages")
PROC_DIR.mkdir(parents=True, exist_ok=True)

if 'dataset_root' not in dir() or dataset_root is None:
    print("WARNING: dataset_root not set. Run Section 3 first.")
else:
    student_pdf_dir = dataset_root / "Student_Pdf"
    student_pdfs = sorted(student_pdf_dir.glob("*.pdf"), key=_student_sort_key)
    MAX_STUDENTS = 5      # Process first 5 students (set None for all 50)
    MAX_EDGE_PX  = 2000   # caps huge scans cleanly while preserving OCR detail

    target_pdfs = student_pdfs[:MAX_STUDENTS] if MAX_STUDENTS else student_pdfs
    total_pdfs = len(target_pdfs)
    print(f"Found {len(student_pdfs)} student PDF(s); ingesting {total_pdfs}...")

    cached_count, processed_count, failed_pdfs = 0, 0, []

    for i, pdf_path in enumerate(target_pdfs, 1):
        sid = pdf_path.stem
        meta_file = PROC_DIR / f"{sid}_meta.json"

        # Resume from checkpoint if metadata and all page images already exist
        if meta_file.exists():
            try:
                cached_pages = json.loads(meta_file.read_text(encoding="utf-8"))
                if cached_pages and all(Path(p["image_path"]).exists() for p in cached_pages):
                    pdf_ingestion_results.extend(cached_pages)
                    cached_count += 1
                    print(f"  [{i:>2}/{total_pdfs}] {pdf_path.name:<15} -> cached ({len(cached_pages)} pages)")
                    continue
            except Exception:
                pass

        try:
            doc = fitz.open(pdf_path)
            total_pages = doc.page_count
            student_pages = []

            for idx in range(total_pages):
                page = doc[idx]
                zoom = min(150.0 / 72.0, MAX_EDGE_PX / max(page.rect.width, page.rect.height))
                pix = page.get_pixmap(matrix=fitz.Matrix(zoom, zoom), alpha=False)
                arr = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, pix.n)
                bgr = cv2.cvtColor(arr, cv2.COLOR_RGB2BGR if pix.n == 3 else cv2.COLOR_RGBA2BGR)
                gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)

                angle = detect_skew(gray)
                if abs(angle) > 0.1:
                    h, w = gray.shape
                    M = cv2.getRotationMatrix2D((w // 2, h // 2), angle, 1.0)
                    gray = cv2.warpAffine(gray, M, (w, h), flags=cv2.INTER_CUBIC,
                                          borderMode=cv2.BORDER_CONSTANT, borderValue=255)

                gray = cv2.fastNlMeansDenoising(gray, None, h=10, templateWindowSize=7, searchWindowSize=15)
                gray = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(gray)

                img_path = PROC_DIR / f"{sid}_p{idx + 1:02d}.jpg"
                cv2.imwrite(str(img_path), gray, [cv2.IMWRITE_JPEG_QUALITY, 85])

                student_pages.append({
                    "student_id": sid,
                    "filename": pdf_path.name,
                    "page_number": idx + 1,
                    "total_pages": total_pages,
                    "image_path": str(img_path),
                    "skew": angle,
                    "shape": list(gray.shape),
                })

            doc.close()
            meta_file.write_text(json.dumps(student_pages, indent=2), encoding="utf-8")
            pdf_ingestion_results.extend(student_pages)
            processed_count += 1
            print(f"  [{i:>2}/{total_pdfs}] {pdf_path.name:<15} -> processed ({total_pages} pages)")
        except Exception as e:
            failed_pdfs.append((pdf_path.name, str(e)))
            print(f"  [{i:>2}/{total_pdfs}] {pdf_path.name:<15} -> ERROR: {e}")

    print(f"\n[STAGE 1] Ingested {len(pdf_ingestion_results)} page(s) across {processed_count + cached_count}/{total_pdfs} PDF(s) "
          f"(new: {processed_count}, cached: {cached_count}, failed: {len(failed_pdfs)})")


Found 50 student PDF(s); ingesting 5...
  [ 1/5] Student_1.pdf   -> cached (4 pages)
  [ 2/5] Student_2.pdf   -> cached (4 pages)
  [ 3/5] Student_3.pdf   -> cached (4 pages)
  [ 4/5] Student_4.pdf   -> cached (4 pages)
  [ 5/5] Student_5.pdf   -> cached (4 pages)

[STAGE 1] Ingested 20 page(s) across 5/5 PDF(s) (new: 0, cached: 5, failed: 0)


## 7. Credentials

Load OpenRouter API key and model from `.env`.

In [7]:
import os
from dotenv import load_dotenv
load_dotenv()

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY", "")
VISION_MODEL       = os.getenv("MODEL", "nvidia/nemotron-3.5-super-vl:free")
OPENROUTER_URL     = "https://openrouter.ai/api/v1/chat/completions"

print(f"Model : {VISION_MODEL}")
print("API key:", "loaded" if OPENROUTER_API_KEY else "MISSING – set OPENROUTER_API_KEY in .env")


Model : nvidia/nemotron-3.5-super-vl:free
API key: loaded


## 8. Handwritten Answer Extraction

Send each processed page to a vision model via OpenRouter.
Returns structured JSON preserving **question numbers** and **page reference**.
Unreadable text is **never fabricated** – uncertain answers are flagged for human review.


In [8]:
import base64, json, re, time, requests, cv2
from pathlib import Path
from collections import defaultdict

OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"
CACHE_DIR = Path("outputs")
CACHE_DIR.mkdir(exist_ok=True)

EXTRACT_PROMPT = (
    "You are grading a handwritten exam.\n"
    "Return ONLY valid JSON (no markdown fences):\n"
    '{"page": <int>, "answers": [{"q": <int>, "text": <str or null>, '
    '"unreadable": <bool>, "needs_review": <bool>}]}\n'
    "Rules:\n"
    "- If text is unclear: text=null, unreadable=true, needs_review=true.\n"
    "- Never fabricate or guess unreadable content.\n"
    "- Include only questions whose answers are visible on this page.\n"
    "- q must be an integer matching the printed question number."
)

def _redact(msg):
    s = str(msg)
    return s.replace(OPENROUTER_API_KEY, "***") if OPENROUTER_API_KEY else s

def encode_img(rec, max_side=1800):
    if "image_path" in rec and Path(rec["image_path"]).exists():
        img = cv2.imread(rec["image_path"])
    elif "processed" in rec:
        img = rec["processed"]
    else:
        raise FileNotFoundError(f"Missing processed image for {rec.get('student_id')} p{rec.get('page_number')}")
    h, w = img.shape[:2]
    scale = min(1.0, max_side / max(h, w))
    if scale < 1.0:
        img = cv2.resize(img, (int(w * scale), int(h * scale)), interpolation=cv2.INTER_AREA)
    _, buf = cv2.imencode(".jpg", img, [cv2.IMWRITE_JPEG_QUALITY, 75])
    return base64.b64encode(buf.tobytes()).decode()

def parse_json_response(raw_text):
    """Strip markdown fences / surrounding prose and validate extraction JSON."""
    if not isinstance(raw_text, str) or not raw_text.strip():
        raise ValueError("Empty or non-string content")
    cleaned = re.sub(r"```(?:json)?\s*|\s*```", "", raw_text, flags=re.IGNORECASE).strip()
    try:
        data = json.loads(cleaned)
    except json.JSONDecodeError:
        start = cleaned.find("{")
        if start == -1:
            raise
        data, _ = json.JSONDecoder().raw_decode(cleaned[start:])
    if not isinstance(data, dict) or not isinstance(data.get("answers"), list):
        raise ValueError("Missing 'answers' list in JSON")
    valid_ans = []
    for a in data["answers"]:
        if not isinstance(a, dict):
            continue
        try:
            q_num = int(a["q"]) if a.get("q") is not None else None
        except (ValueError, TypeError):
            q_num = None
        if q_num is None:
            continue
        text = a.get("text")
        if text is not None and not isinstance(text, str):
            text = str(text)
        unreadable = bool(a.get("unreadable", text is None))
        needs_review = bool(a.get("needs_review", unreadable))
        valid_ans.append({"q": q_num, "text": None if unreadable else text,
                          "unreadable": unreadable, "needs_review": needs_review})
    data["answers"] = valid_ans
    return data

def extract_page(rec, page_no, max_retries=3):
    """Extract handwritten answers for one page with validation & controlled retries."""
    if not OPENROUTER_API_KEY:
        return {"page": page_no, "page_number": page_no, "status": "failed",
                "answers": [], "error": "OPENROUTER_API_KEY not set"}

    b64 = encode_img(rec)
    max_tok = 8192
    last_err = "unknown error"

    for attempt in range(1, max_retries + 1):
        payload = {
            "model": VISION_MODEL,
            "temperature": 0.0,
            "max_tokens": max_tok,
            "reasoning": {"effort": "low"},
            "messages": [{"role": "user", "content": [
                {"type": "text", "text": EXTRACT_PROMPT + f"\nThis is page {page_no} of the exam."},
                {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{b64}"}}
            ]}]
        }
        content, finish = None, None
        try:
            r = requests.post(
                OPENROUTER_URL, json=payload,
                headers={"Authorization": f"Bearer {OPENROUTER_API_KEY}",
                         "Content-Type": "application/json"},
                timeout=90
            )
            if r.status_code == 429 or r.status_code >= 500:
                retry_hdr = r.headers.get("Retry-After")
                wait = int(retry_hdr) if (retry_hdr and retry_hdr.isdigit()) else min(60, 10 * (2 ** (attempt - 1)))
                last_err = f"HTTP {r.status_code}"
                print(f"    p{page_no} [attempt {attempt}/{max_retries}] {last_err}, waiting {wait}s...")
                time.sleep(wait)
                continue
            if r.status_code != 200:
                try:
                    err_info = r.json().get("error", {})
                    err_msg = err_info.get("message", r.text[:200]) if isinstance(err_info, dict) else str(err_info)
                except Exception:
                    err_msg = r.text[:200]
                last_err = f"HTTP {r.status_code}: {_redact(err_msg)}"
                print(f"    p{page_no} [attempt {attempt}/{max_retries}] {last_err}")
                break

            resp = r.json()
            if not isinstance(resp, dict):
                last_err = "Invalid JSON response type"
                continue
            if "error" in resp:
                err_obj = resp["error"]
                msg = err_obj.get("message", str(err_obj)) if isinstance(err_obj, dict) else str(err_obj)
                last_err = f"API error: {_redact(msg)}"
                print(f"    p{page_no} [attempt {attempt}/{max_retries}] {last_err}")
                time.sleep(5 * attempt)
                continue

            choices = resp.get("choices")
            if not isinstance(choices, list) or not choices:
                last_err = "Missing 'choices' in API response"
                continue

            ch = choices[0] if isinstance(choices[0], dict) else {}
            finish = ch.get("finish_reason")
            msg_obj = ch.get("message") if isinstance(ch.get("message"), dict) else {}
            content = msg_obj.get("content")
            refusal = msg_obj.get("refusal")

            if content is None or not isinstance(content, str) or not content.strip():
                last_err = f"Empty/None content (finish_reason={finish}, refusal={refusal})"
                print(f"    p{page_no} [attempt {attempt}/{max_retries}] {last_err}")
                if finish == "length":
                    max_tok = min(max_tok + 4096, 16384)
                time.sleep(5 * attempt)
                continue

            parsed = parse_json_response(content)
            parsed["page"] = page_no
            parsed["page_number"] = page_no
            parsed["error"] = None
            ans_list = parsed["answers"]
            parsed["status"] = (
                "needs_review"
                if (not ans_list or any(a.get("unreadable") or a.get("needs_review") for a in ans_list))
                else "success"
            )
            return parsed

        except (json.JSONDecodeError, ValueError) as e:
            snippet = repr(content[:100]) if isinstance(content, str) else "None"
            last_err = f"Malformed JSON ({type(e).__name__}: {e}) snippet={snippet}"
            print(f"    p{page_no} [attempt {attempt}/{max_retries}] {last_err}")
            if finish == "length":
                max_tok = min(max_tok + 4096, 16384)
            time.sleep(5 * attempt)
        except requests.exceptions.Timeout:
            last_err = "Request timeout"
            print(f"    p{page_no} [attempt {attempt}/{max_retries}] {last_err}")
            time.sleep(5 * attempt)
        except Exception as e:
            last_err = f"{type(e).__name__}: {_redact(e)}"
            print(f"    p{page_no} [attempt {attempt}/{max_retries}] {last_err}")
            time.sleep(5 * attempt)

    return {"page": page_no, "page_number": page_no, "status": "failed",
            "answers": [], "error": last_err}

def save_student_cache(cache_path, sid, filename, total_pages, pages_dict):
    ordered = [pages_dict[k] for k in sorted(pages_dict, key=int)]
    payload = {"student_id": sid, "filename": filename, "total_pages": total_pages, "pages": ordered}
    with open(cache_path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, ensure_ascii=False)

# ── Run per-page extraction across all students with checkpointing ─────────
all_extractions = {}

if 'pdf_ingestion_results' not in dir() or not pdf_ingestion_results:
    print("WARNING: pdf_ingestion_results is empty. Run Section 6 first.")
elif not OPENROUTER_API_KEY:
    print("WARNING: OPENROUTER_API_KEY not set in .env — skipping extraction.")
else:
    pages_by_student = defaultdict(list)
    for r in pdf_ingestion_results:
        pages_by_student[r["student_id"]].append(r)

    INTER_PAGE_DELAY = 2
    total_students = len(pages_by_student)
    total_cached_pages, total_new_pages, total_failed_pages = 0, 0, 0

    for s_idx, (sid, pages) in enumerate(pages_by_student.items(), 1):
        cache_path = CACHE_DIR / f"extractions_{sid}.json"
        pages_dict = {}
        if cache_path.exists():
            try:
                cached = json.loads(cache_path.read_text(encoding="utf-8"))
                for p in cached.get("pages", []):
                    pno = p.get("page_number", p.get("page"))
                    if pno is not None:
                        pages_dict[str(pno)] = p
            except Exception as e:
                print(f"  Warning: could not read cache {cache_path.name}: {e}")

        s_cached, s_new, s_fail = 0, 0, []
        for rec in pages:
            pno = rec["page_number"]
            existing = pages_dict.get(str(pno))
            if existing and existing.get("status") in ("success", "needs_review") and isinstance(existing.get("answers"), list):
                s_cached += 1
                continue

            res = extract_page(rec, pno)
            res["student_id"] = sid
            res["filename"] = rec["filename"]
            pages_dict[str(pno)] = res
            save_student_cache(cache_path, sid, rec["filename"], rec["total_pages"], pages_dict)

            if res["status"] == "failed":
                s_fail.append(pno)
            else:
                s_new += 1
            if pno < rec["total_pages"]:
                time.sleep(INTER_PAGE_DELAY)

        # Retry any failed pages for this student individually
        if s_fail:
            rec_by_pno = {r["page_number"]: r for r in pages}
            still_failed = []
            for pno in s_fail:
                print(f"  [{s_idx}/{total_students}] {sid} retrying failed page {pno}...")
                time.sleep(5)
                res = extract_page(rec_by_pno[pno], pno)
                res["student_id"] = sid
                res["filename"] = rec_by_pno[pno]["filename"]
                pages_dict[str(pno)] = res
                save_student_cache(cache_path, sid, res["filename"], rec_by_pno[pno]["total_pages"], pages_dict)
                if res["status"] == "failed":
                    still_failed.append(pno)
                else:
                    s_new += 1
            s_fail = still_failed

        combined = [
            pages_dict[k] for k in sorted(pages_dict, key=int)
            if pages_dict[k].get("status") in ("success", "needs_review")
        ]
        all_extractions[sid] = combined
        n_ans = sum(len(p.get("answers", [])) for p in combined)
        total_cached_pages += s_cached
        total_new_pages += s_new
        total_failed_pages += len(s_fail)

        print(f"[{s_idx:>2}/{total_students}] {sid:<11} | pages: {len(combined)}/{len(pages)} "
              f"(cached:{s_cached}, new:{s_new}, failed:{len(s_fail)}) | answers: {n_ans}")

    total_ans = sum(len(r.get("answers", [])) for exs in all_extractions.values() for r in exs)
    print(f"\n[STAGE 2] {len(all_extractions)} student(s), {total_ans} answer(s) "
          f"| pages cached:{total_cached_pages}, new:{total_new_pages}, failed:{total_failed_pages}")
    if total_ans == 0:
        print("WARNING: 0 answers extracted. Check API status and logs above.")


[ 1/5] Student_1   | pages: 4/4 (cached:4, new:0, failed:0) | answers: 41
[ 2/5] Student_2   | pages: 4/4 (cached:4, new:0, failed:0) | answers: 29
[ 3/5] Student_3   | pages: 4/4 (cached:4, new:0, failed:0) | answers: 33
[ 4/5] Student_4   | pages: 4/4 (cached:1, new:3, failed:0) | answers: 32
    p2 [attempt 1/3] Empty/None content (finish_reason=length, refusal=None)
[ 5/5] Student_5   | pages: 4/4 (cached:0, new:4, failed:0) | answers: 31

[STAGE 2] 5 student(s), 166 answer(s) | pages cached:13, new:7, failed:0


## 9. Rubric-based Semantic Scoring

Score extracted answers against the answer key via the LLM.
Supports **partial marks**, concept matching, and equivalent terminology.
Keyword-only matching is **not** used for short answers.


In [ ]:
import csv, json, re, time, requests
from pathlib import Path

ak_path = next(Path(".").rglob("answerkey.txt"))
answer_key = {}
with open(ak_path, encoding="utf-8") as f:
    for row in csv.DictReader(f):
        q = int(row["Question_Number"])
        answer_key[q] = {"type": row["Type"].strip(), "answer": row["Correct_Answer"].strip()}

# Load question stems from Question.txt so semantic scoring aligns with both question and answer key
question_texts = {}
q_path = next(Path(".").rglob("Question.txt"), None)
if q_path:
    q_raw = q_path.read_text(encoding="utf-8")
    for m in re.finditer(r"^(\d+)\.\s+(.+?)(?=\n\d+\.\s|\Z)", q_raw, re.MULTILINE | re.DOTALL):
        question_texts[int(m.group(1))] = " ".join(m.group(2).split())

MAX_MARKS = {q: (1.0 if v["type"] == "MCQ" else 2.0) for q, v in answer_key.items()}
SCORING_VERSION = "v2"
print(f"Answer key: {len(answer_key)} questions ({sum(1 for v in answer_key.values() if v['type']=='MCQ')} MCQ, {sum(1 for v in answer_key.values() if v['type']=='Short_Answer')} short) | Max total: {sum(MAX_MARKS.values()):.0f}")

RUBRIC_RULES = (
    "Rubric (max 2.0, in 0.5 steps):\n"
    "- 2.0: Core concept(s) and required parts match model answer (equivalent/informal wording OK).\n"
    "- 1.5: Main concept correct with minor detail missing.\n"
    "- 1.0: Half of a two-part answer (e.g., 1 of 2 items/steps) or partially correct core idea.\n"
    "- 0.5: Minimal relevant concept.\n"
    "- 0.0: Off-topic, wrong, blank, or null/unreadable.\n"
    "- Do NOT invent extra criteria absent from the question or model answer."
)

SCORE_PROMPT = (
    "You are an exam grader. Score the student answer against the question and model answer.\n"
    "Return ONLY valid JSON: "
    '{"q": <int>, "awarded": <float>, "max": <float>, "reasoning": "<=20 words>"}\n' + RUBRIC_RULES
)

BATCH_SCORE_PROMPT = (
    "You are an exam grader. Score each student short answer against its question and model answer.\n"
    "Return ONLY a valid JSON array (no markdown fences):\n"
    '[{"q": <int>, "awarded": <float>, "max": <float>, "reasoning": "<=20 words>"}]\n' + RUBRIC_RULES
)

def _extract_mcq_letter(text):
    """Extract normalized single option letter A-D (handles 'B.', 'B)', 'Option B', 'B - text')."""
    if not text:
        return None
    s = str(text).strip()
    # Reject ambiguous multi-option answers like 'A or B' or 'A, B'
    if re.search(r"\b[A-D]\b\s*(?:,|or|/|&)\s*\b[A-D]\b", s, re.IGNORECASE):
        return None
    m = re.match(r"^(?:option\s*)?([A-D])(?:[\s\.\)\:\-]|$)", s, re.IGNORECASE)
    return m.group(1).upper() if m else None

def _snap_marks(val, max_m, q_type):
    v = max(0.0, min(float(max_m), float(val)))
    if q_type == "MCQ":
        return 1.0 if v >= 0.5 else 0.0
    return round(v * 2.0) / 2.0

def _call_score_api(prompt, max_tokens=2048, retries=3):
    for attempt in range(1, retries + 1):
        try:
            r = requests.post(
                OPENROUTER_URL,
                json={
                    "model": VISION_MODEL,
                    "temperature": 0.0,
                    "max_tokens": max_tokens,
                    "reasoning": {"effort": "low"},
                    "messages": [{"role": "user", "content": prompt}],
                },
                headers={"Authorization": f"Bearer {OPENROUTER_API_KEY}",
                         "Content-Type": "application/json"},
                timeout=60,
            )
            if r.status_code == 429 or r.status_code >= 500:
                retry_hdr = r.headers.get("Retry-After")
                wait = int(retry_hdr) if (retry_hdr and retry_hdr.isdigit()) else min(60, 10 * (2 ** (attempt - 1)))
                time.sleep(wait)
                continue
            r.raise_for_status()
            resp = r.json()
            msg = (resp.get("choices") or [{}])[0].get("message") or {}
            raw = msg.get("content")
            if not isinstance(raw, str) or not raw.strip():
                max_tokens = min(max_tokens + 2048, 8192)
                time.sleep(3 * attempt)
                continue
            return re.sub(r"```(?:json)?\s*|\s*```", "", raw, flags=re.IGNORECASE).strip()
        except Exception:
            time.sleep(3 * attempt)
    return None

def score_one(q, text):
    ak = answer_key.get(q)
    if not ak:
        return {"q": q, "awarded": 0.0, "max": 0.0, "reasoning": "not in answer key"}
    max_m = MAX_MARKS[q]
    if text is None or not str(text).strip():
        return {"q": q, "awarded": 0.0, "max": max_m, "reasoning": "unreadable or unanswered – 0 marks"}
    if ak["type"] == "MCQ":
        chosen = _extract_mcq_letter(text)
        expected = ak["answer"].strip().upper()
        ok = (chosen == expected)
        return {"q": q, "awarded": 1.0 if ok else 0.0, "max": 1.0,
                "reasoning": "correct" if ok else f"got {chosen or text!r}, expected {expected}"}
    if not OPENROUTER_API_KEY:
        return {"q": q, "awarded": 0.0, "max": max_m, "reasoning": "no API key"}
    q_stem = question_texts.get(q, "")
    prompt = (f"{SCORE_PROMPT}\nQ{q}: {q_stem}\nModel answer: {ak['answer']}\n"
              f"Student answer: {text}\nMax marks: {max_m}")
    cleaned = _call_score_api(prompt, max_tokens=2048)
    if cleaned:
        try:
            start = cleaned.find("{")
            result, _ = json.JSONDecoder().raw_decode(cleaned[start:] if start != -1 else cleaned)
            awarded = _snap_marks(result.get("awarded", 0.0), max_m, ak["type"])
            return {"q": q, "awarded": awarded, "max": max_m, "reasoning": str(result.get("reasoning", ""))}
        except Exception as e:
            return {"q": q, "awarded": 0.0, "max": max_m, "reasoning": f"parse error: {e}"}
    return {"q": q, "awarded": 0.0, "max": max_m, "reasoning": "scoring API failed"}

def score_short_batch(items):
    """Score (q, text) pairs in one call; handles duplicate candidates via tag key."""
    if not items or not OPENROUTER_API_KEY:
        return {}
    lines = []
    for idx, (q, text) in enumerate(items):
        q_stem = question_texts.get(q, "")
        lines.append(
            f"[Item {idx}] Q{q} (max {MAX_MARKS[q]}): Question='{q_stem}' | "
            f"Model='{answer_key[q]['answer']}' | Student='{text}'"
        )
    prompt = BATCH_SCORE_PROMPT + "\n\n" + "\n".join(lines)
    cleaned = _call_score_api(prompt, max_tokens=4096)
    scored_by_q = {}
    if cleaned:
        try:
            start = cleaned.find("[")
            arr, _ = json.JSONDecoder().raw_decode(cleaned[start:] if start != -1 else cleaned)
            if isinstance(arr, list):
                for item in arr:
                    if isinstance(item, dict) and item.get("q") is not None:
                        q_num = int(item["q"])
                        if q_num in answer_key and answer_key[q_num]["type"] == "Short_Answer":
                            max_m = MAX_MARKS[q_num]
                            awarded = _snap_marks(item.get("awarded", 0.0), max_m, "Short_Answer")
                            cand = {"q": q_num, "awarded": awarded, "max": max_m,
                                    "reasoning": str(item.get("reasoning", ""))}
                            if q_num not in scored_by_q or cand["awarded"] > scored_by_q[q_num]["awarded"]:
                                scored_by_q[q_num] = cand
        except Exception:
            pass
    for q, text in items:
        if q not in scored_by_q:
            cand = score_one(q, text)
            if q not in scored_by_q or cand["awarded"] > scored_by_q[q]["awarded"]:
                scored_by_q[q] = cand
    return scored_by_q

def verify_scoring_consistency(scores_by_student):
    """Verify every student has all 35 questions scored consistently vs answerkey.txt."""
    issues = []
    expected_qs = set(answer_key.keys())
    for sid, scores in scores_by_student.items():
        q_seen = [s["q"] for s in scores]
        if set(q_seen) != expected_qs or len(q_seen) != len(expected_qs):
            issues.append(f"{sid}: question set mismatch ({len(q_seen)}/{len(expected_qs)})")
        for s in scores:
            q = s["q"]
            ak = answer_key.get(q)
            if not ak:
                issues.append(f"{sid} Q{q}: not in answer key")
                continue
            exp_max = MAX_MARKS[q]
            if s["max"] != exp_max:
                issues.append(f"{sid} Q{q}: max={s['max']} != {exp_max}")
            if not (0.0 <= s["awarded"] <= exp_max):
                issues.append(f"{sid} Q{q}: awarded={s['awarded']} out of bounds [0, {exp_max}]")
            if ak["type"] == "MCQ" and s["awarded"] not in (0.0, 1.0):
                issues.append(f"{sid} Q{q}: invalid MCQ mark {s['awarded']}")
            if ak["type"] == "Short_Answer" and (s["awarded"] * 2) % 1 != 0:
                issues.append(f"{sid} Q{q}: invalid half-mark {s['awarded']}")
    return issues

# ── Score all students with v2 checkpointing ──────────────────────────────
all_scores = {}
SCORE_CACHE_DIR = Path("outputs")

if 'all_extractions' not in dir() or not all_extractions:
    print("WARNING: all_extractions is empty. Run Section 8 first.")
else:
    total_ans_in = sum(len(r.get("answers", [])) for exs in all_extractions.values() for r in exs)
    if total_ans_in == 0:
        print("WARNING: all_extractions has 0 answers. Nothing to score.")
    else:
        n_students = len(all_extractions)
        for s_idx, (sid, extractions) in enumerate(all_extractions.items(), 1):
            sc_cache = SCORE_CACHE_DIR / f"scores_{sid}.json"
            if sc_cache.exists():
                try:
                    raw_cache = json.loads(sc_cache.read_text(encoding="utf-8"))
                    if isinstance(raw_cache, dict) and raw_cache.get("version") == SCORING_VERSION:
                        cached_scores = raw_cache.get("scores", [])
                        if len(cached_scores) == len(answer_key) and not any(
                            "scoring API failed" in str(x.get("reasoning", "")) or "parse error" in str(x.get("reasoning", ""))
                            for x in cached_scores
                        ):
                            all_scores[sid] = cached_scores
                            tot = sum(s["awarded"] for s in cached_scores)
                            print(f"[{s_idx:>2}/{n_students}] {sid:<11} -> cached ({tot:.1f}/{sum(MAX_MARKS.values()):.0f} marks)")
                            continue
                except Exception:
                    pass

            # Collect candidate answers per valid question number (preserves duplicates across pages)
            candidates_by_q = {}
            ignored_qs = []
            for page_res in extractions:
                pno = page_res.get("page", page_res.get("page_number", "?"))
                for ans in page_res.get("answers", []):
                    q = ans.get("q")
                    if q is None:
                        continue
                    if q not in answer_key:
                        ignored_qs.append(q)
                        continue
                    txt = ans.get("text")
                    cands = candidates_by_q.setdefault(q, [])
                    if txt is not None and str(txt).strip():
                        if not any(c["text"] == txt for c in cands):
                            cands.append({"text": txt, "page": pno})
                    elif not cands:
                        cands.append({"text": None, "page": pno})

            scores_by_q = {}
            short_batch_items = []
            extra_short_cands = []
            short_page_lookup = {}

            # Score MCQs and unanswered questions; queue primary Short_Answer for batching + extra duplicates separately
            for q in sorted(answer_key):
                ak = answer_key[q]
                cands = candidates_by_q.get(q, [])
                non_empty = [c for c in cands if c["text"] is not None and str(c["text"]).strip()]

                if not non_empty:
                    scores_by_q[q] = {
                        "q": q, "awarded": 0.0, "max": MAX_MARKS[q],
                        "reasoning": "unreadable or unanswered – 0 marks",
                        "page": cands[0]["page"] if cands else None,
                    }
                elif ak["type"] == "MCQ":
                    best = None
                    for c in non_empty:
                        sc = score_one(q, c["text"])
                        sc["page"] = c["page"]
                        if best is None or sc["awarded"] > best["awarded"]:
                            best = sc
                    scores_by_q[q] = best
                else:
                    short_batch_items.append((q, non_empty[0]["text"]))
                    short_page_lookup[q] = non_empty[0]["page"]
                    for c in non_empty[1:]:
                        extra_short_cands.append((q, c["text"], c["page"]))

            if short_batch_items:
                batch_res = score_short_batch(short_batch_items)
                for q, sc in batch_res.items():
                    sc["page"] = short_page_lookup.get(q)
                    scores_by_q[q] = sc
                for q, txt, pno in extra_short_cands:
                    cand_sc = score_one(q, txt)
                    cand_sc["page"] = pno
                    if cand_sc["awarded"] > scores_by_q[q]["awarded"]:
                        scores_by_q[q] = cand_sc
                time.sleep(1.5)

            scores = [scores_by_q[q] for q in sorted(answer_key)]
            all_scores[sid] = scores
            sc_cache.write_text(
                json.dumps({"version": SCORING_VERSION, "scores": scores}, indent=2, ensure_ascii=False),
                encoding="utf-8",
            )
            tot = sum(s["awarded"] for s in scores)
            extra = f" (ignored out-of-key: {sorted(set(ignored_qs))})" if ignored_qs else ""
            print(f"[{s_idx:>2}/{n_students}] {sid:<11} -> scored ({tot:.1f}/{sum(MAX_MARKS.values()):.0f} marks){extra}")

        issues = verify_scoring_consistency(all_scores)
        print(f"\n[STAGE 3] Scored {len(all_scores)} student(s), {sum(len(v) for v in all_scores.values())} question(s)")
        if issues:
            print("CONSISTENCY WARNINGS:", issues)
        else:
            print(f"[CONSISTENCY CHECK] PASSED ({len(all_scores)} student(s) x {len(answer_key)} questions, max {sum(MAX_MARKS.values()):.0f} marks, 0 violations)")

## 10. Confidence Assessment

Assign **high / medium / low** confidence per scored question using structural signals:
OCR readability, answer completeness, question mapping, and scoring consistency.
Flags uncertain cases for manual review.

> LLM self-reported confidence is NOT treated as a calibrated probability.


In [ ]:
import pandas as pd

def assess_confidence(ext_ans, score):
    signals = []
    if ext_ans is None:
        return "low", True, "not extracted from pages"
    if ext_ans.get("unreadable"):
        return "low", True, "text unreadable"
    if ext_ans.get("needs_review"):
        signals.append("flagged_by_extractor")
    text = ext_ans.get("text") or ""
    if score.get("max", 0) == 2 and len(text.split()) < 3:
        signals.append("very_short_answer")
    if score.get("q") not in answer_key:
        return "low", True, "question not in key"
    if score.get("awarded", 0) > score.get("max", 0):
        return "low", True, "awarded > max (error)"
    if not signals:
        return "high", False, "all signals clear"
    if len(signals) == 1:
        return "medium", False, f"minor: {signals[0]}"
    return "low", True, f"multiple signals: {', '.join(signals)}"

rows = []
for sid, scores in all_scores.items():
    ext_by_q = {}
    for page_res in all_extractions.get(sid, []):
        for ans in page_res.get("answers", []):
            q = ans.get("q")
            if q is not None and (q not in ext_by_q or (ext_by_q[q].get("text") is None and ans.get("text") is not None)):
                ext_by_q[q] = ans
    for sc in scores:
        q = sc["q"]
        ans = ext_by_q.get(q)
        conf, flag, reason = assess_confidence(ans, sc)
        rows.append({
            "student_id": sid, "q": q, "page": sc.get("page"),
            "text": ((ans or {}).get("text") or "")[:40],
            "awarded": sc["awarded"], "max": sc["max"],
            "confidence": conf, "needs_review": flag, "reason": reason,
        })

if rows:
    conf_df = pd.DataFrame(rows)
    print(f"Total assessed questions : {len(conf_df)}")
    print("Confidence distribution:")
    print(conf_df["confidence"].value_counts().to_string())
    flagged_df = conf_df[conf_df["needs_review"]]
    print(f"\nFlagged for manual review: {len(flagged_df)} / {len(conf_df)}")
    if len(flagged_df):
        print("\nSample flagged items (first 15):")
        print(flagged_df[["student_id", "q", "page", "awarded", "max", "confidence", "reason"]].head(15).to_string(index=False))
else:
    conf_df = pd.DataFrame()
    print("No scored data. Run Sections 8 and 9 first.")

## 11. Validation – Automated vs Teacher Marks

Compare automated totals against `Teacher_manual_marks_Anonymized.csv`.
Computes **MAE**, exact agreement, per-question error, and student-wise score difference.
Missing values and mismatched student IDs are handled safely.

In [ ]:
import re
import pandas as pd, numpy as np
from pathlib import Path

marks_path = next(Path(".").rglob("Teacher_manual_marks_Anonymized.csv"))
teacher_df = pd.read_csv(marks_path)
teacher_df["Student_ID"] = teacher_df["Student_ID"].astype(str).str.strip()

auto_totals = {sid.strip(): sum(s["awarded"] for s in scores)
               for sid, scores in all_scores.items()} if 'all_scores' in dir() else {}

if not auto_totals:
    print("WARNING: all_scores is empty — cannot validate. Run Sections 8 & 9 first.")
else:
    if all(t == 0.0 for t in auto_totals.values()):
        print("WARNING: All automated totals are 0.0 — extraction/scoring likely failed.")

    auto_df = pd.DataFrame(auto_totals.items(), columns=["Student_ID", "auto_total"])
    merged = (teacher_df[["Student_ID", "total"]]
              .merge(auto_df, on="Student_ID", how="outer")
              .rename(columns={"total": "teacher_total"}))
    merged["teacher_total"] = pd.to_numeric(merged["teacher_total"], errors="coerce")
    merged["auto_total"]    = pd.to_numeric(merged["auto_total"], errors="coerce")
    merged["diff"] = merged["auto_total"] - merged["teacher_total"]
    merged["_sort"] = merged["Student_ID"].apply(lambda s: int(m.group(1)) if (m := re.search(r"(\d+)", str(s))) else 9999)
    merged = merged.sort_values("_sort").drop(columns=["_sort"]).reset_index(drop=True)

    valid = merged.dropna(subset=["teacher_total", "auto_total"])
    mae   = valid["diff"].abs().mean() if len(valid) else float("nan")
    exact = int((valid["diff"].abs() < 0.5).sum()) if len(valid) else 0
    pct   = 100.0 * exact / len(valid) if len(valid) else 0.0

    print(f"Students matched       : {len(valid)} / {len(merged)}")
    print(f"Unmatched IDs          : {int(merged[['teacher_total', 'auto_total']].isna().any(axis=1).sum())}")
    print(f"MAE (total score)      : {mae:.2f}")
    print(f"Exact agreement (±0.5) : {exact}/{len(valid)} ({pct:.1f}%)")

    if len(valid):
        print("\nMatched student score comparison:")
        print(valid.to_string(index=False))

    sa_cols = [str(q) for q in range(21, 36) if str(q) in teacher_df.columns]
    pq_rows = [{"Student_ID": sid.strip(), "q": str(s["q"]), "auto": s["awarded"]}
               for sid, scores in all_scores.items() for s in scores]
    if pq_rows:
        pq_df = pd.DataFrame(pq_rows)
        pq_pivot = pq_df.pivot_table(index="Student_ID", columns="q", values="auto", aggfunc="first")
        t_sa = teacher_df.set_index("Student_ID")[sa_cols].apply(pd.to_numeric, errors="coerce")
        common = [c for c in pq_pivot.columns if c in t_sa.columns]
        if common:
            pq_mae = (pq_pivot[common] - t_sa[common]).abs().mean().dropna().sort_values(ascending=False)
            print("\nPer-question MAE (short answers):")
            print(pq_mae.round(2).to_string())


## 12. Export Results

Write final results to `outputs/`:
- `results.json`: student-wise totals, evaluation metrics, and per-question evaluations
- `results.csv`: per-question awarded/max marks, confidence, review flag, and reasoning
- `student_marks.csv`: student-wise marks summary compared against teacher marks

In [ ]:
import json, csv, math
import pandas as pd
from pathlib import Path

out_dir = Path("outputs")
out_dir.mkdir(exist_ok=True)

if 'all_scores' not in dir() or not all_scores:
    print("WARNING: all_scores is empty — nothing to export. Run Sections 8 & 9 first.")
else:
    rows = []
    for sid, scores in all_scores.items():
        conf_lookup = {}
        if "conf_df" in dir() and conf_df is not None and len(conf_df):
            for _, r in conf_df[conf_df["student_id"] == sid].iterrows():
                conf_lookup[r["q"]] = {
                    "confidence": r["confidence"],
                    "needs_review": bool(r["needs_review"]),
                    "reason": r["reason"],
                }
        for s in scores:
            q = s["q"]
            ci = conf_lookup.get(q, {"confidence": "unknown", "needs_review": False, "reason": ""})
            rows.append({
                "student_id": sid,
                "question": q,
                "awarded": s.get("awarded", 0.0),
                "max_marks": s.get("max", 0.0),
                "reasoning": s.get("reasoning", ""),
                "confidence": ci["confidence"],
                "needs_review": ci["needs_review"],
                "review_reason": ci["reason"],
            })

    teacher_lookup = {}
    if "merged" in dir() and merged is not None and len(merged):
        for _, r in merged.iterrows():
            if pd.notna(r["teacher_total"]):
                teacher_lookup[str(r["Student_ID"]).strip()] = float(r["teacher_total"])

    student_totals = {}
    for r in rows:
        sid = r["student_id"]
        st = student_totals.setdefault(sid, {"awarded": 0.0, "max": 0.0, "flagged": 0})
        st["awarded"] += r["awarded"]
        st["max"]     += r["max_marks"]
        if r["needs_review"]:
            st["flagged"] += 1

    summaries = []
    for sid, v in student_totals.items():
        awarded = round(v["awarded"], 2)
        t_tot = teacher_lookup.get(sid)
        summaries.append({
            "student_id": sid,
            "total_awarded": awarded,
            "total_max": round(v["max"], 2),
            "percentage": round(100 * v["awarded"] / v["max"], 1) if v["max"] else 0.0,
            "teacher_total": t_tot,
            "score_diff": round(awarded - t_tot, 2) if t_tot is not None else None,
            "flagged_questions": v["flagged"],
        })

    metrics = {}
    if "mae" in dir() and not (isinstance(mae, float) and math.isnan(mae)):
        metrics["MAE_total_score"] = round(float(mae), 4)
    if "exact" in dir() and "valid" in dir() and len(valid):
        metrics["exact_agreement_count"] = int(exact)
        metrics["exact_agreement_pct"]   = round(float(pct), 2)
        metrics["students_matched"]      = int(len(valid))

    doc = {
        "summary": summaries,
        "evaluation_metrics": metrics,
        "per_question_results": rows,
    }

    json_path = out_dir / "results.json"
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(doc, f, indent=2, ensure_ascii=False)

    csv_path = out_dir / "results.csv"
    fieldnames = ["student_id", "question", "awarded", "max_marks",
                  "reasoning", "confidence", "needs_review", "review_reason"]
    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        w.writerows(rows)

    sum_csv_path = out_dir / "student_marks.csv"
    pd.DataFrame(summaries).to_csv(sum_csv_path, index=False)

    # Validate written JSON & CSV
    assert len(json.loads(json_path.read_text(encoding="utf-8"))["per_question_results"]) == len(rows)
    assert len(pd.read_csv(csv_path)) == len(rows)

    print(f"Saved {json_path} ({json_path.stat().st_size} bytes)")
    print(f"Saved {csv_path} ({csv_path.stat().st_size} bytes, {len(rows)} rows)")
    print(f"Saved {sum_csv_path} ({len(summaries)} students)")
